Env setup and import

In [ ]:
import torch
from torch import nn
from env_simplified import make_env
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import MultiSyncCollector, Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
env = make_env()

cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


Flattened Policy

In [13]:
# class CastToFloat(nn.Module):
#     def forward(self, x):
#         return x.float()   # or .to(torch.float32)

# policy_net = nn.Sequential(
#     nn.Flatten(-2),
#     CastToFloat(),
#     MultiAgentMLP(
#         n_agent_inputs = 29 * 34,       
#         n_agent_outputs = 75,       
#         n_agents = 4,
#         centralized = False,        
#         share_params = True,      
#         depth = 8,               
#         num_cells = 1024,
#         activation_class=torch.nn.Tanh
#     )
# )

# # 1. 讀取原始權重
# raw_state_dict = torch.load('actor_net_4.pth')

# # 2. 獲取你目前 policy_net 真正需要的 Keys 列表
# model_keys = list(policy_net.state_dict().keys())

# # 3. 過濾掉檔案中重複的鍵，只按順序保留獨一無二的權重
# #（利用 dict.fromkeys 保持順序並去重，移除多餘的 params 或 _empty_net 命名空間干擾）
# unique_raw_keys = list(dict.fromkeys([k.replace('params', '_empty_net') for k in raw_state_dict.keys()]))
# # 還原回檔案中實際存在的正確鍵名
# src_keys = []
# for k in unique_raw_keys:
#     if k in raw_state_dict:
#         src_keys.append(k)
#     else:
#         src_keys.append(k.replace('_empty_net', 'params'))

# # 4. 建立一對一的對齊字典
# aligned_state_dict = {}
# for m_key, s_key in zip(model_keys, src_keys):
#     aligned_state_dict[m_key] = raw_state_dict[s_key]
#     # 打印對齊狀態供你確認，成功後可刪除此行
#     print(f"對齊成功: {s_key} -> {m_key}")
    
# policy_net.load_state_dict(aligned_state_dict)

# policy_module = TensorDictModule(
#     policy_net,
#     in_keys=[("agents", "observation", "observation")],
#     out_keys=[("agents", "logits")],
# )

# policy = ProbabilisticActor(
#     module=policy_module,
#     spec=env.action_spec_unbatched,
#     in_keys={
#         'logits': ('agents', 'logits'),
#         'mask': ('agents', 'action_mask')
#     }, # type: ignore
#     out_keys=[env.action_key],
#     distribution_class=MaskedCategorical,
#     return_log_prob=True
# )  # we'll need the log-prob for the PPO loss

New policy



In [ ]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

class Unsqueeze(nn.Module):
    def forward(self, x):
        return x.unsqueeze(-3)

policy_net = nn.Sequential(
    CastToFloat(),
    Unsqueeze(),
    MultiAgentConvNet(
        n_agents=4,
        centralized=False,
        share_params=True,
        num_cells=[32, 32, 32],
        paddings=1,
        strides=1,
        kernel_sizes=3,
    ),
    MultiAgentMLP(
        n_agents=4,
        n_agent_inputs=None,
        n_agent_outputs=75,
        num_cells=256,
        centralized=False,
        share_params=True,
        depth=2,
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss

In [ ]:
# import torch
# from torch import nn, Tensor
# from typing import *
# from functools import partial

# # ---------------------------------------------------------------------
# # 1. Core building blocks (unchanged)
# # ---------------------------------------------------------------------

# class ChannelAttention(nn.Module):
#     def __init__(self, channels, ratio=16, actv_builder=nn.ReLU, bias=True):
#         super().__init__()
#         self.shared_mlp = nn.Sequential(
#             nn.Linear(channels, channels // ratio, bias=bias),
#             actv_builder(),
#             nn.Linear(channels // ratio, channels, bias=bias),
#         )
#         if bias:
#             for mod in self.modules():
#                 if isinstance(mod, nn.Linear):
#                     nn.init.constant_(mod.bias, 0)

#     def forward(self, x: Tensor):
#         avg_out = self.shared_mlp(x.mean(-1))
#         max_out = self.shared_mlp(x.amax(-1))
#         weight = (avg_out + max_out).sigmoid()
#         x = weight.unsqueeze(-1) * x
#         return x


# class ResBlock(nn.Module):
#     def __init__(
#         self,
#         channels,
#         *,
#         norm_builder=nn.Identity,
#         actv_builder=nn.ReLU,
#         pre_actv=False,
#     ):
#         super().__init__()
#         self.pre_actv = pre_actv

#         if pre_actv:
#             self.res_unit = nn.Sequential(
#                 norm_builder(),
#                 actv_builder(),
#                 nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
#                 norm_builder(),
#                 actv_builder(),
#                 nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
#             )
#         else:
#             self.res_unit = nn.Sequential(
#                 nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
#                 norm_builder(),
#                 actv_builder(),
#                 nn.Conv1d(channels, channels, kernel_size=3, padding=1, bias=False),
#                 norm_builder(),
#             )
#             self.actv = actv_builder()
#         self.ca = ChannelAttention(channels, actv_builder=actv_builder, bias=True)

#     def forward(self, x):
#         out = self.res_unit(x)
#         out = self.ca(out)
#         out = out + x
#         if not self.pre_actv:
#             out = self.actv(out)
#         return out


# class ResNet(nn.Module):
#     def __init__(
#         self,
#         in_channels,
#         conv_channels,
#         num_blocks,
#         seq_len,  # <-- NEW: pass the sequence length here
#         *,
#         norm_builder=nn.Identity,
#         actv_builder=nn.ReLU,
#         pre_actv=False,
#     ):
#         super().__init__()

#         blocks = []
#         for _ in range(num_blocks):
#             blocks.append(ResBlock(
#                 conv_channels,
#                 norm_builder=norm_builder,
#                 actv_builder=actv_builder,
#                 pre_actv=pre_actv,
#             ))

#         layers = [nn.Conv1d(in_channels, conv_channels, kernel_size=3, padding=1, bias=False)]
#         if pre_actv:
#             layers += [*blocks, norm_builder(), actv_builder()]
#         else:
#             layers += [norm_builder(), actv_builder(), *blocks]
#         layers += [
#             nn.Conv1d(conv_channels, 32, kernel_size=3, padding=1),
#             actv_builder(),
#             nn.Flatten(),
#             # NOW using dynamic seq_len instead of hardcoded 34
#             nn.Linear(32 * seq_len, 1024),
#         ]
#         self.net = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.net(x)


# class UnifiedQNetwork(nn.Module):
#     def __init__(
#         self,
#         in_channels: int,
#         action_space: int,
#         seq_len: int,  # <-- NEW: pass sequence length here
#         conv_channels: int = 256,
#         num_blocks: int = 4,
#         norm_momentum: float = 0.01,
#         norm_eps: float = 1e-3,
#         actv_builder: Optional[Callable[[], nn.Module]] = None,
#         pre_actv: bool = True,
#     ):
#         super().__init__()

#         if actv_builder is None:
#             actv_builder = partial(nn.Mish, inplace=True)

#         norm_builder = partial(nn.BatchNorm1d, conv_channels, momentum=norm_momentum, eps=norm_eps)

#         self.encoder = ResNet(
#             in_channels=in_channels,
#             conv_channels=conv_channels,
#             num_blocks=num_blocks,
#             seq_len=seq_len,  # <-- Pass it down
#             norm_builder=norm_builder,
#             actv_builder=actv_builder,
#             pre_actv=pre_actv,
#         )

#         self.actv = actv_builder()
#         self.fc_q = nn.Linear(1024, action_space)
#         self._freeze_bn = False

#     def forward(self, obs: Tensor) -> Tensor:
#         phi = self.encoder(obs)
#         phi = self.actv(phi)
#         q_values = self.fc_q(phi)
#         return q_values

# # ---------------------------------------------------------------------
# # 2. Your exact configuration + test run
# # ---------------------------------------------------------------------

# if __name__ == "__main__":
#     # Your exact instantiation, now with seq_len=29
#     model = UnifiedQNetwork(
#         in_channels=34,
#         action_space=75,
#         seq_len=52,         # <-- Set to 29!
#         conv_channels=256,
#         num_blocks=4,
#     )

#     # Your exact input shape
#     obs = torch.randn(8, 34, 52)   # batch=8, channels=34, seq_len=29

#     # Forward pass - no more shape errors!
#     output = model(obs)
#     print(f"Input shape:  {obs.shape}")
#     print(f"Output shape: {output.shape}")  # Expected: torch.Size([8, 75])

#     # Optional: count parameters
#     total_params = sum(p.numel() for p in model.parameters())
#     print(f"Total parameters: {total_params:,}")

Input shape:  torch.Size([8, 34, 52])
Output shape: torch.Size([8, 75])
Total parameters: 3,443,883


Critic

In [15]:
# critic_net = nn.Sequential(
#     nn.Flatten(-2),                    # [248,46] -> [248*46]
#     CastToFloat(),
#     nn.Linear(52 * 42, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),                # output 4 values (one per agent)
#     nn.ReLU(),
#     nn.Linear(512, 512),                # output 4 values (one per agent)
#     nn.ReLU(),
#     nn.Linear(512, 4),                # output 4 values (one per agent)
#     nn.Unflatten(-1, (4, 1))          # reshape from [4] to [4,1]
# )
# # 1. 載入原始的 state_dict
# state_dict = torch.load('critic_net_new_1.pth')

# # 2. 移除所有鍵值（Keys）開頭的 "module." 前綴
# from collections import OrderedDict
# new_state_dict = OrderedDict()
# for k, v in state_dict.items():
#     name = k.replace("module.", "") # 如果開頭有 module. 就拿掉
#     new_state_dict[name] = v

# # 3. 將乾淨的 state_dict 載入到你的 critic_net
# critic_net.load_state_dict(new_state_dict)

# critic = TensorDictModule(
#     module=critic_net,
#     in_keys=["state"],               # global state
#     out_keys=[("agents", "state_value")],  # shape [4] values under agents
# )

In [ ]:
class UnsqueezeV2(nn.Module):
    def forward(self, x):
        return x.unsqueeze(-3)
critic_net = nn.Sequential(
    CastToFloat(),
    UnsqueezeV2(),
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(32),
    nn.ReLU(),
    # nn.Dropout2d(0.5),  # Dropout2d is recommended for conv layers (channel-wise)
    
    # Block 2
    nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(64),
    nn.ReLU(),
    # nn.Dropout2d(0.5),
    
    # Block 3
    nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1),
    # nn.BatchNorm2d(128),
    nn.ReLU(),
    # nn.Dropout2d(0.5),
    
    # MLP Head (standard MultiAgentMLP style after flatten)
    nn.Flatten(-3),
    nn.Linear(69888, 512),  
    nn.ReLU(),
    nn.Linear(512, 4),
    nn.Unflatten(-1, (4, 1))
)
critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],               # global state
    out_keys=[("agents", "state_value")],  # shape [4] values under agents
)

Initialize the model

In [ ]:
policy = policy.to('cpu')
critic = critic.to('cpu')
policy(env.reset())
_ =critic(env.reset())

Load models

In [ ]:
policy_net.load_state_dict(torch.load('actor_net_24072026_2.pth'))
critic.load_state_dict(torch.load("critic_net_24072026_2.pth"))
# policy_module= torch.compile(policy_module)
# critic = torch.compile(critic)

<All keys matched successfully>

Rollout

In [23]:
import pygame
from pygame_visualizer import render_game_state

pygame.init()
data = env.rollout(200, policy=policy.to('cpu'))
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
from sys import exit
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
    screen.fill('white')
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()


SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Loss function and optimizer

In [30]:
policy = policy.to(device)
loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic,
    entropy_coeff=0.01
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 5e-5
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

tensor([ 0.1298,  0.1055, -0.1323, -0.0148, -0.2646, -0.0913, -0.2601,  0.2660,
         0.0868,  0.1366, -0.1484, -0.1456, -0.0039,  0.0306, -0.1020,  0.0565,
        -0.1928, -0.1824,  0.2504, -0.1981,  0.0721, -0.0281, -0.1170,  0.1099,
        -0.0479, -0.1987,  0.1664,  0.2480,  0.2252, -0.0504,  0.3067,  0.2017],
       device='cuda:0')

Train loop

In [26]:
num_epochs = 5
max_grad_norm = 0.1
frames_per_batch = 1000  # Number of team frames collected per training iteration
n_iters = 300 # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 1000

if __name__ == "__main__":
    replay_buffer = ReplayBuffer(
        storage=LazyTensorStorage(
            frames_per_batch, device=device
        ),  # We store the frames_per_batch collected at each iteration
        sampler=SamplerWithoutReplacement(),
        batch_size=minibatch_size,  # We will sample minibatches of this siz
    )
    policy=policy.to(device)
    collector= Collector(
        env,
        policy=policy,
        device='cpu',
        storing_device=device,
        frames_per_batch=frames_per_batch,
        total_frames=total_frames
    )

    from tqdm.auto import tqdm
    for it, tensordict_data in enumerate(tqdm(collector)):
        tensordict_data.set(
            ("next", "agents", "done"),
            tensordict_data.get(("next", "done"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        tensordict_data.set(
            ("next", "agents", "terminated"),
            tensordict_data.get(("next", "terminated"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

        with torch.no_grad():
            GAE(
                tensordict_data,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params,
            )  # Compute GAE and add it to the data

        data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
        replay_buffer.extend(data_view)

        for _ in range(num_epochs):
            for _ in range(frames_per_batch // minibatch_size):
                subdata = replay_buffer.sample()
                subdata = subdata.to(device)
                loss_vals = loss_module(subdata)

                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                loss_value.backward()

                torch.nn.utils.clip_grad_norm_(
                    loss_module.parameters(), max_grad_norm
                )  # Optional

                optim.step()
                optim.zero_grad()

        collector.update_policy_weights_()


    torch.save(policy_net.state_dict(), "actor_net_24072026_2.pth")
    torch.save(critic.state_dict(), "critic_net_24072026_2.pth")

C:\Users\ctc73\AppData\Local\Temp\ipykernel_7092\1113423722.py:17: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector= Collector(
c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_ou

Yaku counted (with values):
  tsumo: 1
  hai_di_lao_yue: 1
  fan_pai: 1
2026-07-24 20:48:07,232 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1000]) shape [END]


  6%|▌         | 18/300 [12:36<3:14:57, 41.48s/it]

Yaku counted (with values):
  flowers (combined): 1
  tsumo: 1
  hai_di_lao_yue: 1
  fan_pai: 1
  qing_hun_yi_se: 3


 12%|█▏        | 37/300 [25:45<3:01:55, 41.50s/it]

...

 93%|█████████▎| 279/300 [3:20:59<14:26, 41.26s/it]

Yaku counted (with values):
  flowers (combined): 1
  tsumo: 1
  fan_pai: 1


 99%|█████████▉| 298/300 [3:33:53<01:21, 40.69s/it]

Yaku counted (with values):
  flowers (combined): 1
  tsumo: 1
  fan_pai: 1


100%|██████████| 300/300 [3:35:15<00:00, 43.05s/it]


Rollout per step

In [28]:
import pygame
from sys import exit
from pygame_visualizer import render_game_state
from torchrl.envs.utils import step_mdp  # <-- Added missing utility

pygame.init()
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)

td = env.reset()
policy = policy.to('cpu')

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
            
        # Step through the game manually by pressing SPACE
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            # 1. Check if the game is already over before stepping
            if td.get("done", torch.tensor([False])).any():
                print("Episode finished! Resetting environment...")
                td = env.reset()
                continue
                
            # 2. Policy reads root "observation" and writes root "action" into td
            td = policy(td)
            
            # 3. Environment executes the action and creates the "next" sub-TensorDict
            td = env.step(td)
            
            # 4. Check if this new step ended the game
            if td[("next", "done")].any():
                print(f"Game Over! Reward: {td[('next', 'agents', 'reward')]}")
            
            # 5. Move "next" keys to root level so the policy can read them next turn
            td = step_mdp(td)
            
    screen.fill('white')
    # Render the internal unwrapped environment game state
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_out = _set_tensor_dict(


Game Over! Reward: tensor([[0.],
        [0.],
        [0.],
        [0.]])
Episode finished! Resetting environment...
Game Over! Reward: tensor([[0.],
        [0.],
        [0.],
        [0.]])
Episode finished! Resetting environment...


SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
